In [95]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, confusion_matrix, accuracy_score, f1_score

In [96]:
data = pd.read_csv("loan_approval_data.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Applicant_ID        950 non-null    float64
 1   Applicant_Income    950 non-null    float64
 2   Coapplicant_Income  950 non-null    float64
 3   Employment_Status   950 non-null    object 
 4   Age                 950 non-null    float64
 5   Marital_Status      950 non-null    object 
 6   Dependents          950 non-null    float64
 7   Credit_Score        950 non-null    float64
 8   Existing_Loans      950 non-null    float64
 9   DTI_Ratio           950 non-null    float64
 10  Savings             950 non-null    float64
 11  Collateral_Value    950 non-null    float64
 12  Loan_Amount         950 non-null    float64
 13  Loan_Term           950 non-null    float64
 14  Loan_Purpose        950 non-null    object 
 15  Property_Area       950 non-null    object 
 16  Educati

In [97]:
numerical_col = data.select_dtypes(include = ["number"]).columns
categorical_col = data.select_dtypes(include = ["object"]).columns

In [98]:
num_imp = SimpleImputer(strategy = "mean")
data[numerical_col] = num_imp.fit_transform(data[numerical_col])

In [99]:
cat_imp = SimpleImputer(strategy = "most_frequent")
data[categorical_col] = cat_imp.fit_transform(data[categorical_col])

In [100]:
data.isnull().sum()
data.head(10)


,Applicant_ID,Applicant_Income,Coapplicant_Income,Employment_Status,Age,Marital_Status,Dependents,Credit_Score,Existing_Loans,DTI_Ratio,Savings,Collateral_Value,Loan_Amount,Loan_Term,Loan_Purpose,Property_Area,Education_Level,Gender,Employer_Category,Loan_Approved
0,1.0,17795.0,1387.000000,Salaried,51.0,Married,0.0,637.0,4.0,0.530000,19403.0,45638.0,16619.0,84.0,Personal,Urban,Not Graduate,Female,Private,No
1,2.0,2860.0,2679.000000,Salaried,46.0,Married,3.0,621.0,2.0,0.300000,2580.0,49272.0,38687.0,48.0,Car,Semiurban,Graduate,Male,Private,No
2,3.0,7390.0,2106.000000,Salaried,25.0,Single,2.0,674.0,4.0,0.200000,13844.0,6908.0,27943.0,72.0,Business,Urban,Graduate,Female,Government,Yes
3,4.0,13964.0,8173.000000,Salaried,40.0,Married,2.0,579.0,3.0,0.310000,9553.0,10844.0,27819.0,60.0,Business,Rural,Graduate,Female,Government,No
4,5.0,13284.0,4223.000000,Self-employed,31.0,Single,2.0,721.0,1.0,0.290000,9386.0,37629.0,12741.0,72.0,Car,Urban,Graduate,Male,Private,Yes
5,6.0,8265.0,4831.000000,Salaried,53.0,Single,1.0,602.0,1.0,0.560000,19522.0,2911.0,9798.0,36.0,Home,Semiurban,Graduate,Male,Unemployed,No
6,7.0,18850.0,2768.000000,Salaried,58.0,Married,0.0,687.0,0.0,0.480000,14635.0,8991.0,26143.0,24.0,Home,Rural,Graduate,Male,Private,No
7,8.0,6426.0,3186.000000,Salaried,47.0,Married,2.0,636.0,4.0,0.347263,671.0,11572.0,33747.0,84.0,Personal,Rural,Graduate,Female,Private,No
8,9.0,16423.0,5082.455789,Salaried,54.0,Married,1.0,729.0,0.0,0.590000,777.0,43066.0,34651.0,36.0,Home,Semiurban,Graduate,Male,Private,No
9,10.0,13363.0,2599.000000,Contract,35.0,Single,3.0,726.0,1.0,0.347263,3022.0,29693.0,22182.0,60.0,Personal,Rural,Graduate,Female,Government,Yes


In [101]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder


le = LabelEncoder()
data["Education_Level"] = le.fit_transform(data["Education_Level"])
data["Loan_Approved"] = le.fit_transform(data["Loan_Approved"])


cols = ["Employment_Status", "Marital_Status", "Loan_Purpose", "Property_Area", "Gender", "Employer_Category"]

ohe = OneHotEncoder(drop = "first", sparse_output = False, handle_unknown = "ignore")
encoded = ohe.fit_transform(data[cols])

encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cols), index=data.index)

data = pd.concat([data.drop(columns=cols), encoded_df], axis=1)

In [102]:
X = data.drop("Loan_Approved", axis = 1)
Y = data["Loan_Approved"]

In [103]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size = 0.2, random_state = 42
)

In [104]:
scale = StandardScaler()
X_train = scale.fit_transform(X_train)
X_test = scale.transform(X_test)

In [112]:
#Logistic Regression
log_model = LogisticRegression(max_iter = 4000)
log_model.fit(X_train, Y_train)
y_pred = log_model.predict(X_test)

print("#Logistic Regression")
print("Precision Score",precision_score(Y_test,y_pred))
print("Recall Score", recall_score(Y_test,y_pred))
print("F1_Score",f1_score(Y_test,y_pred))
print("Accuracy",accuracy_score(Y_test,y_pred))
print("CM", confusion_matrix(Y_test,y_pred))
    

#Logistic Regression
Precision Score 0.7868852459016393
Recall Score 0.7868852459016393
F1_Score 0.7868852459016393
Accuracy 0.87
CM [[126  13]
 [ 13  48]]


In [117]:
#KNN
from sklearn.neighbors import KNeighborsClassifier

n_neigh = [2,3,5,7,9]

for k in n_neigh:
    KNN_model = KNeighborsClassifier(n_neighbors = k)
    KNN_model.fit(X_train, Y_train)
    y_pred_KNN = KNN_model.predict(X_test)
    
    print("N_neighbors", k)
    print("#KNN")
    print("Precision Score",precision_score(Y_test,y_pred_KNN))
    print("Recall Score", recall_score(Y_test,y_pred_KNN))
    print("F1_Score",f1_score(Y_test,y_pred_KNN))
    print("Accuracy",accuracy_score(Y_test,y_pred_KNN))
    print("CM", confusion_matrix(Y_test,y_pred_KNN))
    print("\n")

N_neighbors 2
#KNN
Precision Score 0.6666666666666666
Recall Score 0.29508196721311475
F1_Score 0.4090909090909091
Accuracy 0.74
CM [[130   9]
 [ 43  18]]


N_neighbors 3
#KNN
Precision Score 0.5961538461538461
Recall Score 0.5081967213114754
F1_Score 0.5486725663716814
Accuracy 0.745
CM [[118  21]
 [ 30  31]]


N_neighbors 5
#KNN
Precision Score 0.5957446808510638
Recall Score 0.45901639344262296
F1_Score 0.5185185185185185
Accuracy 0.74
CM [[120  19]
 [ 33  28]]


N_neighbors 7
#KNN
Precision Score 0.6428571428571429
Recall Score 0.4426229508196721
F1_Score 0.5242718446601942
Accuracy 0.755
CM [[124  15]
 [ 34  27]]


N_neighbors 9
#KNN
Precision Score 0.6222222222222222
Recall Score 0.45901639344262296
F1_Score 0.5283018867924528
Accuracy 0.75
CM [[122  17]
 [ 33  28]]




In [119]:
# NAIVE BYES

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train, Y_train)
y_pred_nb = nb_model.predict(X_test)


print("#Navie Bayes")
print("Precision Score",precision_score(Y_test,y_pred_nb))
print("Recall Score", recall_score(Y_test,y_pred_nb))
print("F1_Score",f1_score(Y_test,y_pred_nb))
print("Accuracy",accuracy_score(Y_test,y_pred_nb))
print("CM", confusion_matrix(Y_test,y_pred_nb))

#Navie Bayes
Precision Score 0.8035714285714286
Recall Score 0.7377049180327869
F1_Score 0.7692307692307693
Accuracy 0.865
CM [[128  11]
 [ 16  45]]
